# LeetCode 2129 — Capitalize the Title bằng Python

**Link đề:** https://leetcode.com/problems/capitalize-the-title/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        ...
```

## Mục tiêu học
1. Mẫu xử lý chuỗi kinh điển: **tách từ → biến đổi từng từ → ghép lại** (`split` / `join`).
2. Thao tác chữ hoa/thường: `lower()`, `upper()`, `capitalize()`, `title()` — và vì sao `title()` **không đủ** cho bài này.
3. Tự cài đặt đổi hoa/thường bằng **mã ký tự** `ord` / `chr` (chênh lệch 32), duyệt **không dùng split** với chỉ số đầu từ.
4. Hiểu vì sao chuỗi Python **bất biến** (immutable) → nối bằng list rồi `join` thay vì `+=` lặp.
5. Trace, chứng minh, Time/Space, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và cách hiểu đúng

Cho `title` gồm một hoặc nhiều từ cách nhau **một dấu cách**, mỗi từ gồm chữ cái tiếng Anh (hoa hoặc thường). Biến đổi **từng từ**:

- Từ dài **1 hoặc 2** chữ → **toàn bộ chữ thường**.
- Từ dài **≥ 3** chữ → **chữ đầu hoa**, các chữ còn lại **thường**.

| `title` | Kết quả |
|---|---|
| `"capiTalIze tHe titLe"` | `"Capitalize The Title"` |
| `"First leTTeR of EACH Word"` | `"First Letter of Each Word"` |
| `"i lOve leetcode"` | `"i Love Leetcode"` |

**Constraints:** `1 <= len(title) <= 100`; các từ cách nhau đúng một dấu cách, không có dấu cách đầu/cuối; mỗi từ khác rỗng, chỉ chữ cái.

Chú ý: **cả** các chữ sau chữ đầu cũng phải đổi thành thường (`"leTTeR"` → `"Letter"`), và từ ngắn **kể cả từ đầu câu** cũng thường (`"i"`).

In [1]:
examples = [
    ("capiTalIze tHe titLe", "Capitalize The Title"),
    ("First leTTeR of EACH Word", "First Letter of Each Word"),
    ("i lOve leetcode", "i Love Leetcode"),
]
for t, exp in examples:
    print(f"{t!r:30} -> {exp!r}")

'capiTalIze tHe titLe'         -> 'Capitalize The Title'
'First leTTeR of EACH Word'    -> 'First Letter of Each Word'
'i lOve leetcode'              -> 'i Love Leetcode'


## 2. Nền tảng Python cần dùng

### 2.1. `lower()`, `upper()` — trả về **chuỗi mới**

In [2]:
w = "leTTeR"
print(w.lower(), w.upper(), w)    # w không đổi: chuỗi bất biến

letter LETTER leTTeR


### 2.2. `capitalize()` và `title()`

- `w.capitalize()`: chữ đầu hoa, **phần còn lại thường** — đúng thứ ta cần cho từ dài ≥ 3.
- `s.title()`: viết hoa chữ đầu **mọi** từ — nhưng bài này cần từ ngắn viết thường, nên phải sửa thêm.

In [3]:
print("leTTeR".capitalize())                 # Letter
print("first leTTeR of EACH word".title())   # 'Of' bị viết hoa -> chưa đúng đề

Letter
First Letter Of Each Word


### 2.3. `split()` và `" ".join(list)`

`split(" ")` tách theo dấu cách thành list; `" ".join(words)` nối lại, chèn một dấu cách giữa các phần tử.

In [4]:
words = "i lOve leetcode".split(" ")
print(words)
print(" ".join(words))
print("-".join(["a", "b", "c"]))

['i', 'lOve', 'leetcode']
i lOve leetcode
a-b-c


### 2.4. Slicing `w[0]`, `w[1:]`

`w[0]` là chữ đầu, `w[1:]` là phần còn lại (có thể rỗng). Ghép `w[0].upper() + w[1:].lower()` = tự viết `capitalize()`.

In [5]:
w = "tHe"
print(w[0], w[1:], w[0].upper() + w[1:].lower())
print(repr("a"[1:]))   # chuỗi rỗng, không lỗi

t He The
''


### 2.5. Biểu thức điều kiện `A if cond else B` và list comprehension

In [6]:
for w in ["of", "EACH", "i"]:
    print(w, "->", w.lower() if len(w) <= 2 else w.capitalize())
print([x * 2 for x in [1, 2, 3]])

of -> of
EACH -> Each
i -> i
[2, 4, 6]


### 2.6. `ord` / `chr` — chữ cái là con số

`ord('A') = 65`, `ord('a') = 97`: chữ thường = chữ hoa + 32. `chr` làm ngược lại. Kiểm tra hoa: `'A' <= ch <= 'Z'`.

In [7]:
print(ord("A"), ord("a"), ord("a") - ord("A"))
print(chr(ord("G") + 32), chr(ord("g") - 32))
print("Q" <= "Z", "A" <= "q" <= "Z")

65 97 32
g G
True False


### 2.7. Chuỗi bất biến — vì sao nên gom vào list rồi `join`

Không thể sửa `s[0] = 'X'`. Mỗi lần `s += ch` tạo chuỗi mới (có thể tốn `O(độ dài)`); gom ký tự vào list rồi `"".join` một lần là cách chuẩn.

In [8]:
s = "abc"
try:
    s[0] = "X"
except TypeError as e:
    print("TypeError:", e)
chars = list(s)
chars[0] = "X"
print("".join(chars))

TypeError: 'str' object does not support item assignment
Xbc


## 3. Cách 1 — Trực giác: `title()` rồi sửa các từ ngắn

Người mới hay nghĩ: "viết hoa chữ đầu mỗi từ" → `title()`. Sau đó duyệt lại, từ nào dài ≤ 2 thì `lower()`.

In [9]:
def capitalize_title_titlefix(title: str) -> str:
    words = title.title().split(" ")
    for i in range(len(words)):
        if len(words[i]) <= 2:
            words[i] = words[i].lower()
    return " ".join(words)

for t, exp in examples:
    r = capitalize_title_titlefix(t)
    print(f"{r!r:30} {'OK' if r == exp else 'SAI'}")

'Capitalize The Title'         OK
'First Letter of Each Word'    OK
'i Love Leetcode'              OK


## 4. Nút thắt của cách `title()`

Cách trên **đúng** với input của đề, nhưng có hai điểm yếu:

1. **Làm hai lượt** (title cả chuỗi rồi sửa lại) — thừa công cho từ ngắn.
2. `title()` có **quy tắc riêng** về "đầu từ": mọi ký tự sau một ký tự **không phải chữ** được coi là đầu từ. Ở đây chỉ có chữ cái nên không sao, nhưng nếu đề có `'` hay `-` thì `title()` cho kết quả bất ngờ.

In [10]:
print("they're bill's friends".title())   # 'They'Re Bill'S' -> title() coi ' là ranh giới từ
print("e-mail".title())

They'Re Bill'S Friends
E-Mail


**Cải tiến tự nhiên:** quyết định **ngay cho từng từ** theo độ dài, một lượt duy nhất. Quy tắc gọn:

```text
len(w) <= 2  ->  w.lower()
len(w) >= 3  ->  w.capitalize()      # = w[0].upper() + w[1:].lower()
```

## 5. Bước đột phá — Tách bài toán thành hàm cho **một từ**

Mọi quyết định chỉ phụ thuộc **chính từ đó** (không phụ thuộc vị trí từ trong câu). Vì vậy:

1. Viết `fix_word(w)` cho một từ.
2. Áp lên mọi từ: `[fix_word(w) for w in title.split()]`.
3. Ghép: `" ".join(...)`.

Đây là mẫu **map trên các từ** — gặp rất nhiều trong bài chuỗi.

In [11]:
def fix_word(w: str) -> str:
    return w.lower() if len(w) <= 2 else w.capitalize()

for w in ["capiTalIze", "tHe", "of", "EACH", "i", "Ab", "ABC"]:
    print(f"{w!r:13} (len {len(w)}) -> {fix_word(w)!r}")

'capiTalIze'  (len 10) -> 'Capitalize'
'tHe'         (len 3) -> 'The'
'of'          (len 2) -> 'of'
'EACH'        (len 4) -> 'Each'
'i'           (len 1) -> 'i'
'Ab'          (len 2) -> 'ab'
'ABC'         (len 3) -> 'Abc'


## 6. Lời giải chính — mổ xẻ rồi ghép

### 6.1. Đoạn 1: tách từ

In [12]:
title = "First leTTeR of EACH Word"
words = title.split(" ")
print(words)

['First', 'leTTeR', 'of', 'EACH', 'Word']


### 6.2. Đoạn 2: biến đổi từng từ

In [13]:
fixed = [fix_word(w) for w in words]
print(fixed)

['First', 'Letter', 'of', 'Each', 'Word']


### 6.3. Đoạn 3: ghép lại, rồi gói thành hàm

In [14]:
def capitalize_title(title: str) -> str:
    return " ".join(fix_word(w) for w in title.split(" "))

for t, exp in examples:
    r = capitalize_title(t)
    print(f"{r!r:30} {'OK' if r == exp else 'SAI'}")

'Capitalize The Title'         OK
'First Letter of Each Word'    OK
'i Love Leetcode'              OK


### 6.4. Vì sao đây là `split(" ")` mà không phải `split()`?

Với ràng buộc "đúng một dấu cách", hai cách cho cùng kết quả. `split(" ")` + `" ".join` **bảo toàn** đúng số dấu cách nếu có nhiều dấu cách liền nhau (chuỗi rỗng `""` qua `fix_word` vẫn là `""`), còn `split()` sẽ gộp chúng. Đây là chi tiết đáng nhớ khi đề nói "giữ nguyên khoảng trắng".

In [15]:
t = "hello   world"
print(repr(" ".join(fix_word(w) for w in t.split(" "))), "| với split():", repr(" ".join(fix_word(w) for w in t.split())))

'Hello   World' | với split(): 'Hello World'


## 7. TRACE từng bước

### 7.1. Bảng trace — `"First leTTeR of EACH Word"`

| Từ | len | Quy tắc | Kết quả |
|---|:---:|---|---|
| `First` | 5 | ≥ 3 → capitalize | `First` |
| `leTTeR` | 6 | ≥ 3 → capitalize | `Letter` |
| `of` | 2 | ≤ 2 → lower | `of` |
| `EACH` | 4 | ≥ 3 → capitalize | `Each` |
| `Word` | 4 | ≥ 3 → capitalize | `Word` |

In [16]:
def trace_capitalize(title):
    out = []
    for i, w in enumerate(title.split(" ")):
        if len(w) <= 2:
            new, rule = w.lower(), "<= 2 -> lower"
        else:
            new, rule = w[0].upper() + w[1:].lower(), ">= 3 -> hoa đầu + thường"
        out.append(new)
        print(f"  từ {i}: {w!r:12} len={len(w)}  {rule:26} -> {new!r}")
    res = " ".join(out)
    print("  =>", repr(res))
    return res

trace_capitalize("First leTTeR of EACH Word")

  từ 0: 'First'      len=5  >= 3 -> hoa đầu + thường   -> 'First'
  từ 1: 'leTTeR'     len=6  >= 3 -> hoa đầu + thường   -> 'Letter'
  từ 2: 'of'         len=2  <= 2 -> lower              -> 'of'
  từ 3: 'EACH'       len=4  >= 3 -> hoa đầu + thường   -> 'Each'
  từ 4: 'Word'       len=4  >= 3 -> hoa đầu + thường   -> 'Word'
  => 'First Letter of Each Word'


'First Letter of Each Word'

### 7.2. Nhánh đặc biệt: từ ngắn ở đầu câu, và từ dài đúng 3

In [17]:
trace_capitalize("i lOve leetcode")
print()
trace_capitalize("AB abc A")

  từ 0: 'i'          len=1  <= 2 -> lower              -> 'i'
  từ 1: 'lOve'       len=4  >= 3 -> hoa đầu + thường   -> 'Love'
  từ 2: 'leetcode'   len=8  >= 3 -> hoa đầu + thường   -> 'Leetcode'
  => 'i Love Leetcode'

  từ 0: 'AB'         len=2  <= 2 -> lower              -> 'ab'
  từ 1: 'abc'        len=3  >= 3 -> hoa đầu + thường   -> 'Abc'
  từ 2: 'A'          len=1  <= 2 -> lower              -> 'a'
  => 'ab Abc a'


'ab Abc a'

### 7.3. Trace cách duyệt ký tự không dùng `split` (dùng chỉ số đầu từ)

Ta tìm từng từ bằng hai chỉ số `start` (đầu từ) và `i` (vị trí dấu cách tiếp theo hoặc hết chuỗi). Biết độ dài `i - start` thì quyết định được ký tự `start` viết hoa hay thường.

In [18]:
def trace_scan(title):
    chars = list(title.lower())          # mọi ký tự về thường trước
    n, start = len(chars), 0
    for i in range(n + 1):
        if i == n or chars[i] == " ":
            length = i - start
            if length >= 3:
                chars[start] = chars[start].upper()
            print(f"  từ [{start}, {i}) = {title[start:i]!r:12} len={length} -> {''.join(chars[start:i])!r}")
            start = i + 1
    return "".join(chars)

print("=>", repr(trace_scan("capiTalIze tHe titLe")))

  từ [0, 10) = 'capiTalIze' len=10 -> 'Capitalize'
  từ [11, 14) = 'tHe'        len=3 -> 'The'
  từ [15, 20) = 'titLe'      len=5 -> 'Title'
=> 'Capitalize The Title'


## 8. Code nộp LeetCode (Python 3) — nhiều cách

### Cách A — `title()` rồi sửa từ ngắn

In [19]:
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        words = title.title().split(" ")
        return " ".join(w.lower() if len(w) <= 2 else w for w in words)

print([Solution().capitalizeTitle(t) == e for t, e in examples])

[True, True, True]


### Cách B — `split` + `capitalize` / `lower` + `join` (lời giải chính)

In [20]:
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        return " ".join(w.lower() if len(w) <= 2 else w.capitalize()
                        for w in title.split(" "))

print([Solution().capitalizeTitle(t) == e for t, e in examples])

[True, True, True]


### Cách C — Duyệt ký tự, không `split` (một lượt, `O(1)` list phụ ngoài kết quả)

In [21]:
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        chars = list(title.lower())
        n, start = len(chars), 0
        for i in range(n + 1):
            if i == n or chars[i] == " ":
                if i - start >= 3:
                    chars[start] = chars[start].upper()
                start = i + 1
        return "".join(chars)

print([Solution().capitalizeTitle(t) == e for t, e in examples])

[True, True, True]


### Cách D — Tự cài đặt hoa/thường bằng `ord` / `chr` (không dùng hàm có sẵn)

Hữu ích để hiểu bản chất và cho các ngôn ngữ không có `capitalize`.

In [22]:
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        def to_lower(ch: str) -> str:
            return chr(ord(ch) + 32) if "A" <= ch <= "Z" else ch

        def to_upper(ch: str) -> str:
            return chr(ord(ch) - 32) if "a" <= ch <= "z" else ch

        out = []
        for w in title.split(" "):
            low = [to_lower(c) for c in w]
            if len(w) >= 3:
                low[0] = to_upper(low[0])
            out.append("".join(low))
        return " ".join(out)

print([Solution().capitalizeTitle(t) == e for t, e in examples])

[True, True, True]


### Cách E — Dùng `map` với hàm tách riêng

In [23]:
class Solution:
    def capitalizeTitle(self, title: str) -> str:
        def fix(w: str) -> str:
            return w.capitalize() if len(w) > 2 else w.lower()
        return " ".join(map(fix, title.split(" ")))

print([Solution().capitalizeTitle(t) == e for t, e in examples])

[True, True, True]


## 9. Vì sao thuật toán đúng?

**Tách và ghép không làm sai cấu trúc:** do ràng buộc (một dấu cách, không đầu/cuối), `title.split(" ")` trả đúng danh sách các từ `w1..wk` theo thứ tự, và `" ".join` dựng lại đúng khung `w1 w2 ... wk`. Vì vậy chỉ cần mỗi từ được biến đổi đúng.

**Mỗi từ:** `fix_word` là định nghĩa của đề viết thành code:
- `len ≤ 2` → `lower()` đổi **mọi** chữ thành thường.
- `len ≥ 3` → `capitalize()` = chữ đầu hoa + **mọi** chữ còn lại thường.

**Cách C — invariant:** sau bước `lower()`, mọi ký tự đều thường. Khi gặp ranh giới (dấu cách hoặc hết chuỗi) ở `i`, đoạn `[start, i)` đúng là một từ (vì `start` luôn là vị trí ngay sau dấu cách trước đó). Chỉ ký tự `start` của từ dài ≥ 3 được viết hoa ⇒ đúng quy tắc.

## 10. Phân tích Time / Space Complexity

Gọi `n = len(title)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| A. `title()` + sửa | `O(n)` (hai lượt) | `O(n)` | dễ nghĩ; nhạy với ký tự đặc biệt |
| B. split/capitalize/join | `O(n)` | `O(n)` | lời giải chính |
| C. Duyệt ký tự | `O(n)` | `O(n)` (list ký tự = kết quả) | không tạo list từ |
| D. `ord`/`chr` | `O(n)` | `O(n)` | tự cài đặt |
| E. `map` | `O(n)` | `O(n)` | cùng bản chất B |

Kết quả có độ dài `n` nên `O(n)` bộ nhớ là không tránh được (chuỗi bất biến).

## 11. Kiểm thử: mẫu đề, biên, phản ví dụ, đối chiếu

In [24]:
import random

def sol_A(t): return capitalize_title_titlefix(t)
def sol_B(t): return capitalize_title(t)
def sol_C(t): return trace_free_scan(t)
def trace_free_scan(t):
    chars = list(t.lower()); n, start = len(chars), 0
    for i in range(n + 1):
        if i == n or chars[i] == " ":
            if i - start >= 3: chars[start] = chars[start].upper()
            start = i + 1
    return "".join(chars)
def sol_D(t):
    lo = lambda c: chr(ord(c) + 32) if "A" <= c <= "Z" else c
    up = lambda c: chr(ord(c) - 32) if "a" <= c <= "z" else c
    out = []
    for w in t.split(" "):
        x = [lo(c) for c in w]
        if len(w) >= 3: x[0] = up(x[0])
        out.append("".join(x))
    return " ".join(out)
def sol_E(t): return " ".join(map(lambda w: w.capitalize() if len(w) > 2 else w.lower(), t.split(" ")))

SOLS = [sol_A, sol_B, sol_C, sol_D, sol_E]

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:28} PASS")

tests = examples + [
    ("A", "a"), ("ab", "ab"), ("ABC", "Abc"),          # biên độ dài 1, 2, 3
    ("aBcD", "Abcd"),
    ("L hV", "l hv"),                                   # mọi từ ngắn
    ("ThIS IS A TEST", "This is a Test"),
]
for t, exp in tests:
    for f in SOLS:
        check(f"{t!r} {f.__name__}", f(t), exp)

'capiTalIze tHe titLe' sol_A PASS
'capiTalIze tHe titLe' sol_B PASS
'capiTalIze tHe titLe' sol_C PASS
'capiTalIze tHe titLe' sol_D PASS
'capiTalIze tHe titLe' sol_E PASS
'First leTTeR of EACH Word' sol_A PASS
'First leTTeR of EACH Word' sol_B PASS
'First leTTeR of EACH Word' sol_C PASS
'First leTTeR of EACH Word' sol_D PASS
'First leTTeR of EACH Word' sol_E PASS
'i lOve leetcode' sol_A      PASS
'i lOve leetcode' sol_B      PASS
'i lOve leetcode' sol_C      PASS
'i lOve leetcode' sol_D      PASS
'i lOve leetcode' sol_E      PASS
'A' sol_A                    PASS
'A' sol_B                    PASS
'A' sol_C                    PASS
'A' sol_D                    PASS
'A' sol_E                    PASS
'ab' sol_A                   PASS
'ab' sol_B                   PASS
'ab' sol_C                   PASS
'ab' sol_D                   PASS
'ab' sol_E                   PASS
'ABC' sol_A                  PASS
'ABC' sol_B                  PASS
'ABC' sol_C                  PASS
'ABC' sol_D            

In [25]:
rng = random.Random(2129)
letters = "abcXYZ"
for _ in range(500):
    t = " ".join("".join(rng.choice(letters) for _ in range(rng.randint(1, 6))) for _ in range(rng.randint(1, 6)))
    outs = {f(t) for f in SOLS}
    assert len(outs) == 1, (t, outs)
    out = outs.pop()
    # Tính chất: giữ nguyên độ dài và vị trí dấu cách; chữ (không kể hoa/thường) không đổi
    assert len(out) == len(t) and out.lower() == t.lower()
print("500 test ngẫu nhiên: 5 cách khớp nhau và giữ đúng tính chất")

500 test ngẫu nhiên: 5 cách khớp nhau và giữ đúng tính chất


## 12. Lỗi người mới hay gặp — code sai, vì sao sai, cách sửa

### Lỗi 1: Chỉ viết hoa chữ đầu, quên hạ phần còn lại

In [26]:
def wrong_no_lower_rest(title):
    return " ".join(w if len(w) <= 2 else w[0].upper() + w[1:] for w in title.split(" "))   # SAI

print("Sai :", repr(wrong_no_lower_rest("First leTTeR of EACH Word")))
print("Đúng:", repr(capitalize_title("First leTTeR of EACH Word")))

Sai : 'First LeTTeR of EACH Word'
Đúng: 'First Letter of Each Word'


**Vì sao sai:** đề yêu cầu các chữ còn lại **thường**, và từ ngắn cũng phải `lower()` (ở đây `"of"` tình cờ đã thường). **Sửa:** `w[0].upper() + w[1:].lower()` hoặc `capitalize()`; từ ngắn `lower()`.

### Lỗi 2: Nhầm ngưỡng `< 2` hoặc `<= 3`

In [27]:
def wrong_threshold(title):
    return " ".join(w.lower() if len(w) < 2 else w.capitalize() for w in title.split(" "))   # SAI

print("Sai :", repr(wrong_threshold("i lOve of")), "| Đúng:", repr(capitalize_title("i lOve of")))

Sai : 'i Love Of' | Đúng: 'i Love of'


### Lỗi 3: Dùng `title()` mà không sửa từ ngắn

In [28]:
print("Sai :", repr("First leTTeR of EACH Word".title()), "| Đúng:", repr(capitalize_title("First leTTeR of EACH Word")))

Sai : 'First Letter Of Each Word' | Đúng: 'First Letter of Each Word'


### Lỗi 4: Gọi `w.lower()` mà không gán lại (chuỗi bất biến)

In [29]:
def wrong_no_assign(title):
    words = title.split(" ")
    for w in words:
        if len(w) <= 2:
            w.lower()              # SAI: kết quả bị bỏ đi
        else:
            w = w.capitalize()     # SAI: chỉ gán biến vòng lặp, list không đổi
    return " ".join(words)

print("Sai :", repr(wrong_no_assign("i lOve leetcode")), "| Đúng:", repr(capitalize_title("i lOve leetcode")))

Sai : 'i lOve leetcode' | Đúng: 'i Love Leetcode'


**Sửa:** gán vào đúng phần tử `words[i] = ...` (duyệt bằng chỉ số) hoặc tạo list mới bằng comprehension.

### Lỗi 5: Viết hoa từ đầu câu dù nó ngắn (nhầm với quy tắc tiếng Anh thông thường)

In [30]:
def wrong_first_word(title):
    words = [fix_word(w) for w in title.split(" ")]
    words[0] = words[0].capitalize()       # SAI: đề không có ngoại lệ cho từ đầu
    return " ".join(words)

print("Sai :", repr(wrong_first_word("i lOve leetcode")), "| Đúng:", repr(capitalize_title("i lOve leetcode")))

Sai : 'I Love Leetcode' | Đúng: 'i Love Leetcode'


## 13. Mẹo nhận dạng pattern

- "Biến đổi **từng từ** độc lập" → `" ".join(f(w) for w in s.split(" "))`.
- Quy tắc phụ thuộc **độ dài** / **vị trí** trong từ → tách hàm `f(w)` cho một từ, test riêng.
- Cần sửa ký tự tại chỗ → `list(s)` rồi `"".join` (chuỗi bất biến).
- Cẩn thận với hàm "thông minh" như `title()` — đọc kỹ đề có khớp đúng quy tắc của hàm không.

Bài liên quan: 2114 Maximum Number of Words Found in Sentences, 1816 Truncate Sentence, 557 Reverse Words in a String III, 520 Detect Capital.

## 14. Biến thể

### 14.1. Ngưỡng `k` tùy chọn

In [31]:
def capitalize_title_k(title, k):
    return " ".join(w.lower() if len(w) <= k else w.capitalize() for w in title.split(" "))

print(capitalize_title_k("the lord of the rings", 3))

the Lord of the Rings


### 14.2. Detect Capital (LeetCode 520): một từ có viết hoa "đúng cách" không?

Đúng khi: toàn hoa, toàn thường, hoặc chỉ chữ đầu hoa.

In [32]:
def detect_capital(w):
    return w.isupper() or w.islower() or (w[0].isupper() and w[1:].islower())

print([detect_capital(w) for w in ["USA", "leetcode", "Google", "FlaG", "g"]])

[True, True, True, False, True]


### 14.3. Đảo ký tự trong từng từ (LeetCode 557) — cùng mẫu map trên từ

In [33]:
def reverse_words_iii(s):
    return " ".join(w[::-1] for w in s.split(" "))

print(reverse_words_iii("Let's take LeetCode contest"))

s'teL ekat edoCteeL tsetnoc


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

Đáp án ở **mục 17**.

### Bài 1 — Tự viết `my_capitalize(w)` không dùng `capitalize()`

In [34]:
def my_capitalize(w):
    # TODO: viết code ở đây
    pass

# assert my_capitalize("leTTeR") == "Letter" and my_capitalize("a") == "A"

### Bài 2 — Tự viết lại lời giải #2129

In [35]:
def capitalize_title_ex(title):
    # TODO: viết code ở đây
    pass

# assert capitalize_title_ex("First leTTeR of EACH Word") == "First Letter of Each Word"

### Bài 3 — Đổi hoa ↔ thường mọi chữ (swapcase) bằng `ord`/`chr`

In [36]:
def my_swapcase(s):
    # TODO: viết code ở đây
    pass

# assert my_swapcase("aBc D") == "AbC d"

### Bài 4 — Biến thể: từ dài ≥ 3 viết hoa **chữ cuối** thay vì chữ đầu

In [37]:
def capitalize_last(title):
    # TODO: viết code ở đây
    pass

# assert capitalize_last("hello of WORLD") == "hellO of worlD"

### Bài 5 — Tự tạo test làm lộ **Lỗi 1** (quên hạ phần còn lại)

In [38]:
my_tests = [
    # TODO: các title mà wrong_no_lower_rest(t) != capitalize_title(t)
]
# for t in my_tests:
#     assert wrong_no_lower_rest(t) != capitalize_title(t)

### Bài 6 — Mini challenge: "camelCase" hóa
Biến `"hello big WORLD"` thành `"helloBigWorld"`: từ đầu toàn thường, các từ sau `capitalize()`, bỏ dấu cách.

In [39]:
def to_camel(title):
    # TODO: viết code ở đây
    pass

# assert to_camel("hello big WORLD") == "helloBigWorld" and to_camel("X") == "x"

## 16. Gợi ý ba mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | chữ đầu + phần còn lại | `w[0].upper()` | `+ w[1:].lower()` |
| 2 | tách → sửa → ghép | `fix_word` theo độ dài | `" ".join(...)` |
| 3 | hoa ↔ thường lệch 32 | kiểm tra khoảng `'A'..'Z'` | `chr(ord(c) ± 32)` |
| 4 | như capitalize nhưng chữ cuối | `w[:-1].lower()` | `+ w[-1].upper()` |
| 5 | lỗi giữ nguyên chữ hoa ở giữa | cần từ dài ≥ 3 có chữ hoa sau chữ đầu, hoặc từ ngắn có chữ hoa | `"leTTeR"`, `"OF"` |
| 6 | từ đầu xử lý khác | `words[0].lower()` | `"".join([...] + [w.capitalize() for w in words[1:]])` |

## 17. Đáp án tham khảo

In [40]:
def my_capitalize(w):
    return w[0].upper() + w[1:].lower()

def capitalize_title_ex(title):
    return " ".join(w.lower() if len(w) <= 2 else my_capitalize(w) for w in title.split(" "))

def my_swapcase(s):
    out = []
    for c in s:
        if "A" <= c <= "Z":
            out.append(chr(ord(c) + 32))
        elif "a" <= c <= "z":
            out.append(chr(ord(c) - 32))
        else:
            out.append(c)
    return "".join(out)

def capitalize_last(title):
    return " ".join(w.lower() if len(w) <= 2 else w[:-1].lower() + w[-1].upper() for w in title.split(" "))

my_tests = ["leTTeR", "OF", "aBC"]

def to_camel(title):
    words = title.split(" ")
    return words[0].lower() + "".join(w.capitalize() for w in words[1:])

assert my_capitalize("leTTeR") == "Letter" and my_capitalize("a") == "A"
assert capitalize_title_ex("First leTTeR of EACH Word") == "First Letter of Each Word"
assert my_swapcase("aBc D") == "AbC d"
assert capitalize_last("hello of WORLD") == "hellO of worlD"
for t in my_tests:
    assert wrong_no_lower_rest(t) != capitalize_title(t), t
assert to_camel("hello big WORLD") == "helloBigWorld" and to_camel("X") == "x"
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


**Giải thích bằng lời:**
- Bài 1: `capitalize` = hoa chữ đầu + thường phần còn lại; `w[1:]` rỗng khi `w` một ký tự, không lỗi.
- Bài 2: lời giải chính dùng hàm bài 1.
- Bài 3: chữ hoa và chữ thường cách nhau đúng 32 trong bảng mã ASCII.
- Bài 4: chỉ thay vị trí chữ được viết hoa (`w[-1]`).
- Bài 5: Lỗi 1 để nguyên các chữ hoa sau chữ đầu và không hạ từ ngắn (`"OF"` vẫn là `"OF"`).
- Bài 6: từ đầu là ngoại lệ, các từ sau dùng `capitalize`, ghép không dấu cách.

## 18. Tổng kết — template + checklist

**Template "map trên từng từ":**
```python
return " ".join(f(w) for w in s.split(" "))
# với bài này:
f = lambda w: w.lower() if len(w) <= 2 else w.capitalize()
```

**Checklist trước khi submit:**
- [ ] Từ dài 1–2: **toàn thường** (kể cả từ đầu câu).
- [ ] Từ dài ≥ 3: hoa chữ đầu **và** thường phần còn lại.
- [ ] Ngưỡng đúng: `<= 2` / `>= 3`.
- [ ] Gán lại kết quả (chuỗi bất biến).
- [ ] Không dựa vào `title()` mà không sửa.
- [ ] Đúng tên method `capitalizeTitle`.